In [1]:
"""
Cog-RAG -- Cognitive-Inspired Dual-Hypergraph with Theme Alignment
====================================================================
Reference    : Hu et al., "Cog-RAG: Cognitive-Inspired Dual-Hypergraph with
               Theme Alignment Retrieval-Augmented Generation"
               arXiv: 2511.13201v2  (Nov 2025, updated Dec 2025)

Architecture : FIVE configurations covering the complete Cog-RAG design space:
               (1) Baseline Flat RAG         -- standard chunk FAISS retrieval
               (2) Entity Hypergraph Only     -- fine-grained high-order entity
                                                 relation retrieval with hyperedge
                                                 diffusion, no theme layer
               (3) Theme Hypergraph Only      -- inter-chunk thematic structure
                                                 retrieval, no entity detail layer
               (4) Full Dual-Hypergraph       -- theme HG + entity HG combined,
                   (no cognitive ordering)       flat score fusion (no two-stage
                                                 cognitive retrieval order)
               (5) Full Cog-RAG Two-Stage     -- cognitive top-down two-stage:
                   Cognitive Retrieval [BEST]    Stage 1: theme hypergraph
                                                 activates macro-thematic content
                                                 + generates theme-aware answer;
                                                 Stage 2: entity hypergraph guided
                                                 by theme-aware answer + entity
                                                 keywords performs fine-grained
                                                 diffusion recall

=============================================================================
HYPERGRAPH FUNDAMENTALS -- WHY HYPERGRAPHS AND NOT PLAIN GRAPHS
=============================================================================
A standard graph G = (V, E) encodes BINARY relations: each edge e in E connects
exactly 2 vertices. For knowledge representation this means every relational fact
is decomposed into a set of pairwise assertions:

    Fact:   "Multi-head attention concatenates h=8 parallel attention heads,
            each with d_k=64, projecting to d_model=512"
    Binary decomposition:
        Edge(multi-head attention, h=8)
        Edge(multi-head attention, d_k=64)
        Edge(multi-head attention, d_model=512)
        Edge(h=8, parallel heads)
        ...
    Loss:   the simultaneous constraint that h=8 AND d_k=64 AND d_model=512
            are ALL properties of the SAME mechanism is not representable as
            binary edges. GraphRAG and LightRAG both suffer this fragmentation.

A hypergraph H = (V, E_H) replaces binary edges with HYPEREDGES: each hyperedge
e in E_H can connect ANY number of vertices n >= 2 simultaneously.

    Hyperedge e = {multi-head attention, h=8, d_k=64, d_model=512,
                   parallel heads, concat, projection W^O}
    Description: "Multi-head attention runs h=8 attention heads in parallel,
                  each with d_k = d_model/h = 64 dimensions. Outputs are
                  concatenated and projected through W^O to produce d_model=512."

This preserves the complete n-ary relational fact as a SINGLE queryable unit.
Retrieval of this hyperedge delivers ALL related entities and their constraints
simultaneously, eliminating the reasoning chain breaks that graph-based RAG
introduces by forcing multi-hop traversal to reconstruct what was originally
a single coherent fact.

=============================================================================
COG-RAG DUAL-HYPERGRAPH ARCHITECTURE
=============================================================================
Cog-RAG maintains TWO distinct hypergraphs built from the same document corpus,
modeling knowledge at two complementary granularities:

THEME HYPERGRAPH G_theme = (V_key, E_theme):
    Granularity: INTER-CHUNK, macro-thematic, global.
    Built by:   For each chunk D_i, an LLM extracts:
                    E_theme = LLM(P_ext_theme(D_i))    -> narrative theme label
                    V_key   = LLM(P_ext_key(D_i, E_theme)) -> key entities
                One hyperedge per chunk connecting its key entities under its theme.
    Structure:  Each hyperedge = one document chunk's narrative theme.
                Vertices = key entities that participate in that theme.
    Purpose:    Captures how themes span and connect multiple chunks.
                "The Transformer architecture" theme connects entities across
                the encoder section, decoder section, and the attention paper.
    Query path: Extract theme keywords from query -> match against theme labels
                -> retrieve top-k theme hyperedges -> diffuse to neighboring
                key entities -> retrieve chunk contexts of matched hyperedges.

ENTITY HYPERGRAPH G_entity = (V_entity, E_entity):
    Granularity: INTRA-CHUNK, fine-grained, local.
    Built by:   For each chunk D_i, an LLM extracts ALL multi-entity relations:
                    E_entity_i = LLM(P_ext_entity(D_i))
                Each relation connects 2-N entities co-occurring in a shared fact.
    Structure:  Each hyperedge = one relational fact spanning n >= 2 entities.
                Multiple hyperedges per chunk (one per distinct relational fact).
    Purpose:    Captures high-order co-occurrence and relational constraints
                within individual chunks. Enables fine-grained factoid retrieval.
    Query path: Extract entity keywords from query -> match against entity names
                in V_entity -> score hyperedges by entity coverage + semantic sim
                -> retrieve top-k hyperedges + associated chunk contexts.

COGNITIVE TWO-STAGE RETRIEVAL (mirroring top-down human reasoning):
    Stage 1 (macro-thematic): Activates global thematic context first.
        Human analogy: "Before recalling specific numbers, I recall what the
        Transformer architecture chapter was about."
        Result: theme-aware preliminary answer A_theme (partial, thematic).

    Stage 2 (fine-grained recall guided by Stage 1):
        Human analogy: "Guided by the chapter topic, I now recall the specific
        d_model and h values I read."
        Input: Stage 1's A_theme guides which entities to search for in Stage 2.
        Result: entity-level hyperedge context fused with A_theme for final answer.

=============================================================================
INDEXING COST -- WHY THIS IS AN OFFLINE OPERATION
=============================================================================
Building both hypergraphs requires ONE LLM call per chunk per hypergraph:
    For each chunk: 1 call for theme extraction + 1 call for entity extraction
    Total indexing LLM calls: 2 * n_chunks

For our 3 arXiv PDFs (~1,200 chunks at 450 chars):
    Indexing: ~2,400 LLM calls (gpt-4o-mini at ~$0.15/1M input tokens)
    Estimated cost: ~2,400 * 300 input tokens * $0.00000015 = ~$0.11

Indexing is done ONCE and the hypergraphs are persisted (JSON + FAISS).
Query-time cost: ~3 LLM calls for Config 5:
    1. keyword extraction (fast, ~100 tokens)
    2. theme-aware preliminary answer (Stage 1, ~500 tokens)
    3. final answer (Stage 2, ~800 tokens)

=============================================================================
IMPLEMENTATION NOTE ON HYPEREDGE STORAGE
=============================================================================
This production implementation represents the hypergraph using:
    - A Python dict mapping hyperedge_id -> HyperEdge dataclass
    - A FAISS index over hyperedge DESCRIPTION embeddings (BGE-large)
    - An incidence mapping: entity_name -> [hyperedge_id, ...]
    - Chunk index: hyperedge_id -> original Document chunk

This avoids heavy graph database dependencies (Neo4j, networkx large-scale).
For production corpora >100K chunks: use a proper hypergraph DB or
neo4j with hyperedge-as-node pattern (each hyperedge becomes a node,
entities become edges connecting the hyperedge node to entity nodes).


"""


'\nCog-RAG -- Cognitive-Inspired Dual-Hypergraph with Theme Alignment\n====================================================================\nReference    : Hu et al., "Cog-RAG: Cognitive-Inspired Dual-Hypergraph with\n               Theme Alignment Retrieval-Augmented Generation"\n               arXiv: 2511.13201v2  (Nov 2025, updated Dec 2025)\n\nArchitecture : FIVE configurations covering the complete Cog-RAG design space:\n               (1) Baseline Flat RAG         -- standard chunk FAISS retrieval\n               (2) Entity Hypergraph Only     -- fine-grained high-order entity\n                                                 relation retrieval with hyperedge\n                                                 diffusion, no theme layer\n               (3) Theme Hypergraph Only      -- inter-chunk thematic structure\n                                                 retrieval, no entity detail layer\n               (4) Full Dual-Hypergraph       -- theme HG + entity HG combined,\n   

In [2]:

import os
import sys
import json
import time
import pickle
import logging
import hashlib
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Set, Any

import numpy as np


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma,FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0525 21:51:32.307000 18648 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:

# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("cograg")


In [5]:


# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class CogRAGConfig:
    """
    Centralized configuration for the Cog-RAG pipeline.

    THEME_TOP_K (5):
        Number of top theme hyperedges retrieved by Stage 1 (theme keyword matching).
        Each theme hyperedge represents the macro-theme of one document chunk.
        At k=5, Stage 1 activates 5 thematic perspectives on the query.
        These 5 themes span the global structure: for a query about attention
        mechanisms, themes might include "self-attention", "multi-head attention",
        "positional encoding", "encoder-decoder attention", and "transformer architecture".

    ENTITY_TOP_K (6):
        Number of top entity hyperedges retrieved by Stage 2 (entity keyword + guided search).
        Each entity hyperedge is a fine-grained relational fact within a chunk.
        At k=6, Stage 2 delivers 6 specific factual relationships to supplement
        the thematic context from Stage 1.

    DIFFUSION_HOPS (1):
        In Cog-RAG's theme hypergraph, after retrieving the top-k theme hyperedges,
        a diffusion step expands to neighboring vertices (key entities) connected
        to those hyperedges. Diffusion hops = 1 means we expand one step outward:
        from the retrieved hyperedge's vertices, we also activate hyperedges that
        SHARE a vertex with the retrieved set.
        The diffusion effect: queries about "attention heads" activate the
        "multi-head attention" theme hyperedge, whose diffusion then activates
        the "positional encoding" hyperedge if they share the "embedding dimension"
        entity. This models the cognitive "activation spread" of human recall.

    ENTITY_COVERAGE_WEIGHT (0.6) and SEMANTIC_WEIGHT (0.4):
        Entity hyperedge scoring formula (Stage 2):
            score(e) = ENTITY_COVERAGE_WEIGHT * coverage(e, query_entities)
                     + SEMANTIC_WEIGHT * cosine_sim(embed(e.description), embed(query))
        Coverage measures what fraction of query entities appear in hyperedge e.
        Semantic sim measures overall alignment of the hyperedge description with the query.
        0.6/0.4 weighting prioritizes entity coverage over semantic similarity
        because in fine-grained entity retrieval, finding the exact entities mentioned
        in the query is more important than broad semantic alignment.
        For conceptual queries: lower ENTITY_COVERAGE_WEIGHT (0.4) and higher
        SEMANTIC_WEIGHT (0.6) may improve results.

    MAX_CONTEXT_CHARS (3600):
        Maximum characters of retrieved context fed to the LLM.
        3600 chars ~= 900 tokens. At typical Azure OpenAI pricing:
        900 context tokens * ~$0.005/1K = $0.0045 per query context.
        Increasing to 7200 chars doubles cost but improves recall on complex queries.

    INDEX_CACHE_DIR:
        Directory for persisting both hypergraphs across runs.
        Contains: theme_hypergraph.json, entity_hypergraph.json,
                  faiss_theme_index/, faiss_entity_index/
        If index files exist and PDF fingerprint matches, indexing is skipped.
        If PDFs change (fingerprint mismatch), full re-indexing is triggered.
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",     "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers",  "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",   "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- Index Persistence -------------------------------------------------
    INDEX_CACHE_DIR: str = "./cograg_index"
    FAISS_FLAT_DIR:  str = "./faiss_cograg_flat"

    # --- BGE Embeddings (bi-encoder for all embedding tasks) ---------------
    BIENCODER_MODEL:             str = "BAAI/bge-large-en-v1.5"
    BIENCODER_DEVICE:            str = "cpu"
    BIENCODER_QUERY_INSTRUCTION: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # --- Chunking ----------------------------------------------------------
    CHUNK_SIZE:    int = 450
    CHUNK_OVERLAP: int = 60

    # --- Hypergraph Construction -------------------------------------------
    MAX_ENTITIES_PER_HYPEREDGE: int = 6    # cap entity list per hyperedge
    MAX_HYPEREDGES_PER_CHUNK:   int = 5    # entity hyperedges per chunk
    THEME_KEYWORDS_PER_THEME:   int = 4    # key entities per theme hyperedge

    # --- Retrieval Parameters ----------------------------------------------
    THEME_TOP_K:    int   = 5    # Stage 1: top theme hyperedges
    ENTITY_TOP_K:   int   = 6    # Stage 2: top entity hyperedges
    FLAT_TOP_K:     int   = 4    # Baseline flat FAISS top-K
    DIFFUSION_HOPS: int   = 1    # theme hypergraph neighborhood diffusion steps

    # Entity hyperedge scoring weights (coverage vs. semantic)
    ENTITY_COVERAGE_WEIGHT: float = 0.6
    SEMANTIC_WEIGHT:        float = 0.4

    # --- Context Assembly --------------------------------------------------
    MAX_CONTEXT_CHARS: int = 3600   # ~900 tokens

    # --- LLM ---------------------------------------------------------------
    LLM_ANSWER_TEMPERATURE:   float = 0.0
    LLM_EXTRACT_TEMPERATURE:  float = 0.0    # deterministic for extraction
    OLLAMA_BASE_URL: str = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    OLLAMA_MODEL:    str = os.environ.get("OLLAMA_MODEL",    "qwen2.5-coder:7b")
    LLM_MAX_TOKENS: int             = 1024

    # --- Prompts -----------------------------------------------------------
    P_EXT_THEME: str = """You are a document analysis assistant.
Given the following document chunk, extract:
1. A concise THEME LABEL (3-7 words): the main narrative theme of this chunk.
2. KEY ENTITIES (up to {max_entities}): the most important named entities, concepts,
   or technical terms central to this theme.

Document chunk:
{chunk_text}

Respond ONLY with valid JSON:
{{"theme": "<theme label>", "key_entities": ["entity1", "entity2", ...]}}"""

    P_EXT_ENTITY: str = """You are a knowledge extraction assistant.
Given the document chunk and its theme, extract high-order relational facts.
Each fact is a HYPEREDGE: a group of 2-{max_entities} entities that share a
meaningful relationship described in one sentence.

Theme: {theme}
Document chunk:
{chunk_text}

Extract up to {max_hyperedges} relational facts. Each fact must:
- Connect 2 or more distinct entities/concepts from the chunk.
- Be expressed as a concise natural language description (1-2 sentences).
- Represent an n-ary relationship (not just a binary pair when possible).

Respond ONLY with valid JSON:
{{"hyperedges": [
  {{"entities": ["e1", "e2", "e3"], "description": "fact description ..."}},
  ...
]}}"""

    P_KEYWORD_EXTRACT: str = """You are a keyword extraction assistant.
Given the following query, extract:
1. THEME KEYWORDS (up to 3): high-level thematic terms that describe the topic.
2. ENTITY KEYWORDS (up to 5): specific named entities, numbers, technical terms.

Query: {query}

Respond ONLY with valid JSON:
{{"theme_keywords": ["kw1", "kw2"], "entity_keywords": ["ent1", "ent2", ...]}}"""

    P_THEME_ANSWER: str = """You are a research assistant.
Using the thematic context below, provide a PRELIMINARY answer that captures
the high-level structure and key themes relevant to the question.
This is a DRAFT answer focused on macro-understanding, not fine details.

Thematic context:
{theme_context}

Question: {question}

Preliminary thematic answer (3-5 sentences, focus on structure and themes):"""

    P_FINAL_ANSWER: str = """You are a precise technical research assistant.
Answer the question using the context below.
The context includes both thematic context (macro) and entity-level facts (micro).
Synthesize both levels into a precise, complete answer.
If the answer is not present, respond:
"The provided documents do not contain sufficient information."

Thematic context (macro-level):
{theme_context}

Entity-level facts (micro-level):
{entity_context}

Preliminary answer to refine: {theme_answer}

Question: {question}

Final precise answer:"""

    P_BASELINE_ANSWER: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer is not present, respond:
"The provided documents do not contain sufficient information to answer this question."

Context:
{context}

Question: {question}

Precise technical answer:"""


In [6]:

# ===========================================================================
# SECTION 2 -- DATA STRUCTURES
# ===========================================================================

@dataclass
class HyperEdge:
    """
    A hyperedge in either the theme or entity hypergraph.

    In the THEME hypergraph:
        hedge_id:    unique identifier (sha256 prefix of chunk_id + theme)
        entities:    key entities extracted from the chunk for this theme
        description: the theme label (e.g., "Multi-head attention mechanism")
        chunk_ids:   [chunk_id] -- the chunk this theme hyperedge belongs to
        hedge_type:  "theme"

    In the ENTITY hypergraph:
        hedge_id:    unique identifier (sha256 prefix of chunk_id + description)
        entities:    the n >= 2 entities participating in this relational fact
        description: the natural language description of the n-ary relation
        chunk_ids:   [chunk_id] -- the chunk this entity hyperedge was extracted from
        hedge_type:  "entity"

    embedding: BGE-large embedding of the description (1024-dim).
               Stored as a flat list for JSON serialization.
               Reconstructed as np.array for FAISS similarity search.
    """
    hedge_id:    str
    entities:    List[str]
    description: str
    chunk_ids:   List[str]
    hedge_type:  str   # "theme" or "entity"
    embedding:   Optional[List[float]] = None


@dataclass
class HyperGraph:
    """
    In-memory hypergraph structure.

    hedges:           dict mapping hedge_id -> HyperEdge
    entity_to_hedges: dict mapping entity_name -> [hedge_id, ...]
                      Used for entity-keyword-based lookup and diffusion.
    chunk_to_hedges:  dict mapping chunk_id -> [hedge_id, ...]
                      Used to retrieve all hyperedges for a given chunk.
    hedge_embeddings: np.array of shape (n_hedges, 1024) -- all embeddings
                      stacked in hedge_id_list order for FAISS search.
    hedge_id_list:    list of hedge_ids in the same order as hedge_embeddings rows.
    """
    hedges:            Dict[str, HyperEdge]    = field(default_factory=dict)
    entity_to_hedges:  Dict[str, List[str]]    = field(default_factory=dict)
    chunk_to_hedges:   Dict[str, List[str]]    = field(default_factory=dict)
    hedge_embeddings:  Optional[np.ndarray]    = None
    hedge_id_list:     List[str]               = field(default_factory=list)

    def add_hyperedge(self, hedge: HyperEdge) -> None:
        self.hedges[hedge.hedge_id] = hedge
        for ent in hedge.entities:
            ent_lower = ent.lower().strip()
            if ent_lower not in self.entity_to_hedges:
                self.entity_to_hedges[ent_lower] = []
            if hedge.hedge_id not in self.entity_to_hedges[ent_lower]:
                self.entity_to_hedges[ent_lower].append(hedge.hedge_id)
        for cid in hedge.chunk_ids:
            if cid not in self.chunk_to_hedges:
                self.chunk_to_hedges[cid] = []
            self.chunk_to_hedges[cid].append(hedge.hedge_id)

    def build_embedding_matrix(self) -> None:
        """Stack all hyperedge embeddings into a matrix for batch FAISS search."""
        ids = []
        vecs = []
        for hid, hedge in self.hedges.items():
            if hedge.embedding is not None:
                ids.append(hid)
                vecs.append(hedge.embedding)
        if vecs:
            self.hedge_id_list    = ids
            self.hedge_embeddings = np.array(vecs, dtype=np.float32)

    def n_hedges(self) -> int:
        return len(self.hedges)

    def neighbor_hedges(self, hedge_ids: List[str]) -> Set[str]:
        """
        One-hop diffusion: find all hyperedges that share at least one entity
        with any of the input hedge_ids. Used for Stage 1 diffusion in Cog-RAG.

        Args:
            hedge_ids: list of hyperedge IDs to diffuse from.

        Returns:
            Set of hyperedge IDs reachable in 1 hop (excluding input set).
        """
        seed_entities: Set[str] = set()
        for hid in hedge_ids:
            hedge = self.hedges.get(hid)
            if hedge:
                for ent in hedge.entities:
                    seed_entities.add(ent.lower().strip())

        neighbor_ids: Set[str] = set()
        for ent in seed_entities:
            for nhid in self.entity_to_hedges.get(ent, []):
                if nhid not in hedge_ids:
                    neighbor_ids.add(nhid)

        return neighbor_ids


@dataclass
class CogRAGTrace:
    """
    Execution trace for one Cog-RAG query.

    stage1_theme_hedges:  list of theme hyperedge IDs activated in Stage 1.
    stage1_diffused:      set of additional hedge IDs from Stage 1 diffusion.
    stage2_entity_hedges: list of entity hyperedge IDs retrieved in Stage 2.
    theme_keywords:       keywords extracted from query for theme matching.
    entity_keywords:      keywords extracted from query for entity matching.
    theme_answer:         preliminary thematic answer from Stage 1 (Config 5 only).
    """
    question:              str
    strategy:              str
    theme_keywords:        List[str]       = field(default_factory=list)
    entity_keywords:       List[str]       = field(default_factory=list)
    stage1_theme_hedges:   List[str]       = field(default_factory=list)
    stage1_diffused:       Set[str]        = field(default_factory=set)
    stage2_entity_hedges:  List[str]       = field(default_factory=list)
    theme_answer:          str             = ""
    final_answer:          str             = ""
    index_ms:              float           = 0.0   # indexing (only at first run)
    retrieval_ms:          float           = 0.0
    generation_ms:         float           = 0.0
    total_ms:              float           = 0.0
    llm_calls:             int             = 0

    def print_trace(self) -> None:
        print(f"\n{'='*74}")
        print(f"TRACE: {self.strategy}")
        print(f"Query: {self.question[:74]}")
        print(f"{'='*74}")
        if self.theme_keywords:
            print(f"  Theme keywords : {self.theme_keywords}")
        if self.entity_keywords:
            print(f"  Entity keywords: {self.entity_keywords}")
        if self.stage1_theme_hedges:
            print(f"  Stage1 theme hedges activated: {len(self.stage1_theme_hedges)}")
            print(f"  Stage1 diffused neighbors:     {len(self.stage1_diffused)}")
        if self.stage2_entity_hedges:
            print(f"  Stage2 entity hedges activated:{len(self.stage2_entity_hedges)}")
        if self.theme_answer:
            print(f"\n  Preliminary theme answer:\n  {self.theme_answer[:280]}")
        print(f"\n  LLM calls: {self.llm_calls}")
        print(f"  Latency: retrieval={self.retrieval_ms:.0f}ms  "
              f"gen={self.generation_ms:.0f}ms  "
              f"total={self.total_ms:.0f}ms")
        print(f"\n  FINAL ANSWER:\n  {self.final_answer[:450]}")
        print("=" * 74 + "\n")


In [7]:


# ===========================================================================
# SECTION 3 -- INFRASTRUCTURE BUILDERS
# ===========================================================================

def download_pdfs(config: CogRAGConfig) -> List[Path]:
    dl_dir = Path(config.PDF_DOWNLOAD_DIR)
    dl_dir.mkdir(parents=True, exist_ok=True)
    paths: List[Path] = []
    for name, url in config.PDF_SOURCES:
        dest = dl_dir / f"{name}.pdf"
        if dest.exists() and dest.stat().st_size > 10_000:
            logger.info("Cached: %s", dest.name)
            paths.append(dest)
            continue
        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url, headers={"User-Agent": "Mozilla/5.0 (CogRAG/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()
            dest.write_bytes(data)
            paths.append(dest)
            time.sleep(1.0)
        except Exception as exc:
            raise RuntimeError(f"Cannot download '{url}': {exc}") from exc
    return paths


def load_and_chunk_documents(pdf_paths: List[Path], config: CogRAGConfig) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
        add_start_index=True,
    )
    all_chunks: List[Document] = []
    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        pages = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata.update({"paper_name": paper_name, "source": pdf_path.name})
        chunks = splitter.split_documents(pages)
        for i, chunk in enumerate(chunks):
            chunk.metadata["chunk_id"] = f"{pdf_path.stem}_chunk_{i}"
        logger.info("  %s -> %d chunks", pdf_path.name, len(chunks))
        all_chunks.extend(chunks)
    logger.info("Total chunks: %d", len(all_chunks))
    return all_chunks


def build_bge_embeddings(config: CogRAGConfig) -> HuggingFaceEmbeddings:
    logger.info("Loading BGE: %s", config.BIENCODER_MODEL)
    return HuggingFaceEmbeddings(
        model_name=config.BIENCODER_MODEL,
        model_kwargs={"device": config.BIENCODER_DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )


def build_flat_faiss(chunks: List[Document], embeddings: HuggingFaceEmbeddings,
                     config: CogRAGConfig) -> FAISS:
    faiss_file = Path(config.FAISS_FLAT_DIR) / "index.faiss"
    if faiss_file.exists():
        try:
            vs = FAISS.load_local(config.FAISS_FLAT_DIR, embeddings,
                                   allow_dangerous_deserialization=True)
            logger.info("Flat FAISS loaded. Vectors: %d", vs.index.ntotal)
            return vs
        except Exception:
            pass
    logger.info("Building flat FAISS from %d chunks ...", len(chunks))
    vs = FAISS.from_documents(chunks, embeddings)
    Path(config.FAISS_FLAT_DIR).mkdir(parents=True, exist_ok=True)
    vs.save_local(config.FAISS_FLAT_DIR)
    return vs


def build_ollama_llm(config: CogRAGConfig) -> ChatOllama:
    """Connect to local Ollama and return a ChatOllama instance."""
    import urllib.request, urllib.error
    base_url = getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434')
    model    = getattr(config, 'OLLAMA_MODEL',    'qwen2.5-coder:7b')
    try:
        urllib.request.urlopen(base_url, timeout=3)
    except Exception as exc:
        raise RuntimeError(
            f"Ollama not reachable at {base_url}. Start Ollama and run: "
            f"ollama pull qwen2.5-coder:7b"
        ) from exc
    return ChatOllama(
        base_url=base_url,
        model=model,
        temperature=getattr(config, 'LLM_ANSWER_TEMPERATURE', 0.0),
        num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
    )

In [45]:

# ===========================================================================
# SECTION 4 -- HYPERGRAPH CONSTRUCTION (offline indexing)
# ===========================================================================

def _llm_call(prompt_text: str, llm: ChatOllama) -> str:
    """Direct LLM call returning raw string. Used for extraction prompts."""
    return llm.invoke(prompt_text).content


def _safe_json_parse(raw: str, fallback: Any) -> Any:
    """Parse JSON response from LLM, stripping markdown code fences if present."""
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1]) if len(lines) > 2 else cleaned
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        logger.warning("JSON parse failed. Raw: %s ...", cleaned[:120])
        return fallback


def extract_theme_hyperedge(
    chunk: Document, llm: ChatOllama, config: CogRAGConfig,
) -> Optional[HyperEdge]:
    """
    Extract ONE theme hyperedge from a document chunk.

    Per Cog-RAG (Hu et al., 2025) Algorithm 1:
        E_theme = LLM(P_ext_theme(D_i))
        V_key   = LLM(P_ext_key(D_i, E_theme))

    We combine both into a single LLM call for efficiency.
    The result: one HyperEdge with:
        description = theme label (narrative macro-theme of this chunk)
        entities    = key entities under this theme (up to THEME_KEYWORDS_PER_THEME)
        chunk_ids   = [chunk.metadata["chunk_id"]]
        hedge_type  = "theme"

    Args:
        chunk  : Document chunk to extract theme from.
        llm    : ChatOllama for extraction.
        config : CogRAGConfig.

    Returns:
        HyperEdge (theme type) or None if extraction fails.
    """
    prompt = config.P_EXT_THEME.format(
        chunk_text=chunk.page_content[:400],
        max_entities=config.THEME_KEYWORDS_PER_THEME,
    )
    raw = _llm_call(prompt, llm)
    parsed = _safe_json_parse(raw, {"theme": "", "key_entities": []})

    theme_label = str(parsed.get("theme", "")).strip()
    key_entities = [str(e).strip() for e in parsed.get("key_entities", [])
                    if str(e).strip()]

    if not theme_label or not key_entities:
        return None

    chunk_id = chunk.metadata.get("chunk_id", hashlib.sha256(
        chunk.page_content[:50].encode()).hexdigest()[:8])
    hedge_id = hashlib.sha256(
        f"{chunk_id}::{theme_label}".encode()).hexdigest()[:16]

    return HyperEdge(
        hedge_id   = hedge_id,
        entities   = key_entities[: config.THEME_KEYWORDS_PER_THEME],
        description= theme_label,
        chunk_ids  = [chunk_id],
        hedge_type = "theme",
    )


def extract_entity_hyperedges(
    chunk: Document, theme: str, llm: ChatOllama, config: CogRAGConfig,
) -> List[HyperEdge]:
    """
    Extract MULTIPLE entity hyperedges from a document chunk.

    Per Cog-RAG (Hu et al., 2025):
        E_entity_i = LLM(P_ext_entity(D_i))
        Each hyperedge = one n-ary relational fact (n >= 2 entities).

    The entity hypergraph captures HIGH-ORDER relations that binary GraphRAG
    cannot represent. For the Transformer paper chunk:
        "We employ h = 8 parallel attention heads. For each, d_k = d_v = d_model/h = 64."
    Entity hyperedge: entities=["multi-head attention", "h=8 heads", "d_k=64", "d_v=64"]
                      description="Multi-head attention uses 8 parallel heads each
                                   with d_k = d_v = 64 in the base model."

    This is one hyperedge that a binary graph would require 6+ edges to approximate,
    and which those 6 edges still could not fully represent (the joint constraint
    h * d_k = d_model = 512 is not recoverable from individual pairwise edges).

    Args:
        chunk  : Document chunk.
        theme  : Theme label extracted in the theme extraction step.
        llm    : ChatOllama.
        config : CogRAGConfig.

    Returns:
        List of HyperEdge (entity type). Empty list if extraction fails.
    """
    prompt = config.P_EXT_ENTITY.format(
        theme=theme,
        chunk_text=chunk.page_content[:400],
        max_entities=config.MAX_ENTITIES_PER_HYPEREDGE,
        max_hyperedges=config.MAX_HYPEREDGES_PER_CHUNK,
    )
    raw = _llm_call(prompt, llm)
    parsed = _safe_json_parse(raw, {"hyperedges": []})

    chunk_id = chunk.metadata.get("chunk_id", hashlib.sha256(
        chunk.page_content[:50].encode()).hexdigest()[:8])

    hedges: List[HyperEdge] = []
    for hdata in parsed.get("hyperedges", [])[: config.MAX_HYPEREDGES_PER_CHUNK]:
        entities    = [str(e).strip() for e in hdata.get("entities", []) if str(e).strip()]
        description = str(hdata.get("description", "")).strip()
        if len(entities) < 2 or not description:
            continue
        hedge_id = hashlib.sha256(
            f"{chunk_id}::{description[:60]}".encode()).hexdigest()[:16]
        hedges.append(HyperEdge(
            hedge_id   = hedge_id,
            entities   = entities[: config.MAX_ENTITIES_PER_HYPEREDGE],
            description= description,
            chunk_ids  = [chunk_id],
            hedge_type = "entity",
        ))

    return hedges


def embed_hypergraph_descriptions(
    hg: HyperGraph, embeddings: HuggingFaceEmbeddings,
) -> None:
    """
    Embed all hyperedge descriptions in a HyperGraph using BGE-large.

    The description embedding enables semantic similarity search during retrieval:
        - For the theme hypergraph: search by semantic similarity to theme keywords.
        - For the entity hypergraph: search by semantic similarity to entity keywords.

    Embeddings are stored back into each HyperEdge.embedding as a Python list
    (for JSON serialization) and then stacked into HyperGraph.hedge_embeddings
    as a numpy array for efficient cosine search.

    Args:
        hg         : HyperGraph with hedges already added.
        embeddings : HuggingFaceEmbeddings (BGE-large).
    """
    descriptions = [hg.hedges[hid].description for hid in hg.hedges]
    if not descriptions:
        return

    logger.info("  Embedding %d hyperedge descriptions ...", len(descriptions))
    # Batch encode all descriptions at once
    vecs = embeddings.embed_documents(descriptions)

    for i, hid in enumerate(hg.hedges):
        hg.hedges[hid].embedding = vecs[i]

    hg.build_embedding_matrix()


def build_dual_hypergraph(
    chunks:     List[Document],
    embeddings: HuggingFaceEmbeddings,
    llm:        ChatOllama,
    config:     CogRAGConfig,
) -> Tuple[HyperGraph, HyperGraph, Dict[str, Document]]:
    """
    Build both the theme hypergraph and entity hypergraph from the document corpus.

    This is the OFFLINE INDEXING phase of Cog-RAG. It should be run ONCE and
    the resulting hypergraphs persisted to disk. For our 3 arXiv PDFs (~1,200 chunks):
        LLM calls: ~2,400 (2 per chunk: 1 theme + 1 entity extraction)
        Wall time: ~30-45 minutes on CPU with gpt-4o-mini
        Cost:      ~$0.11 at gpt-4o-mini pricing

    The chunk_registry maps chunk_id -> Document for context retrieval at query time.

    Args:
        chunks     : All document chunks from the corpus.
        embeddings : HuggingFaceEmbeddings for description embedding.
        llm        : ChatOllama for extraction.
        config     : CogRAGConfig.

    Returns:
        Tuple of (theme_hg, entity_hg, chunk_registry).
    """
    theme_hg   = HyperGraph()
    entity_hg  = HyperGraph()
    chunk_registry: Dict[str, Document] = {}

    logger.info("Building dual hypergraph from %d chunks ...", len(chunks))
    logger.info("WARNING: This requires ~%d LLM calls. Run once and cache.",
                2 * len(chunks))

    for i, chunk in enumerate(chunks):
        chunk_id = chunk.metadata.get("chunk_id", f"chunk_{i}")
        chunk_registry[chunk_id] = chunk

        if i % 50 == 0:
            logger.info("  Processing chunk %d / %d ...", i, len(chunks))

        # --- Extract theme hyperedge (1 LLM call per chunk) ---------------
        theme_hedge = extract_theme_hyperedge(chunk, llm, config)
        if theme_hedge:
            theme_hg.add_hyperedge(theme_hedge)
            theme_label = theme_hedge.description
        else:
            theme_label = "general"

        # --- Extract entity hyperedges (1 LLM call per chunk) -------------
        entity_hedges = extract_entity_hyperedges(chunk, theme_label, llm, config)
        for e_hedge in entity_hedges:
            entity_hg.add_hyperedge(e_hedge)

    # --- Embed all hyperedge descriptions ---------------------------------
    logger.info("Embedding theme hypergraph (%d hedges) ...", theme_hg.n_hedges())
    embed_hypergraph_descriptions(theme_hg, embeddings)

    logger.info("Embedding entity hypergraph (%d hedges) ...", entity_hg.n_hedges())
    embed_hypergraph_descriptions(entity_hg, embeddings)

    logger.info(
        "Dual hypergraph built: theme=%d hedges, entity=%d hedges",
        theme_hg.n_hedges(), entity_hg.n_hedges(),
    )
    return theme_hg, entity_hg, chunk_registry


def save_hypergraphs(
    theme_hg: HyperGraph, entity_hg: HyperGraph,
    chunk_registry: Dict[str, Document], cache_dir: str,
) -> None:
    """Persist hypergraphs and chunk registry to disk for fast reload."""
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)

    def hg_to_dict(hg: HyperGraph) -> Dict:
        return {
            "hedges": {hid: {
                "hedge_id":    h.hedge_id,
                "entities":    h.entities,
                "description": h.description,
                "chunk_ids":   h.chunk_ids,
                "hedge_type":  h.hedge_type,
                "embedding":   h.embedding,
            } for hid, h in hg.hedges.items()},
            "entity_to_hedges": hg.entity_to_hedges,
            "chunk_to_hedges":  hg.chunk_to_hedges,
            "hedge_id_list":    hg.hedge_id_list,
        }

    with open(cache / "theme_hypergraph.json", "w") as f:
        json.dump(hg_to_dict(theme_hg), f)
    with open(cache / "entity_hypergraph.json", "w") as f:
        json.dump(hg_to_dict(entity_hg), f)

    # Save chunk registry using pickle (Documents contain metadata dicts)
    with open(cache / "chunk_registry.pkl", "wb") as f:
        pickle.dump(chunk_registry, f)

    logger.info("Hypergraphs saved to %s", cache_dir)


def load_hypergraphs(cache_dir: str) -> Optional[Tuple[HyperGraph, HyperGraph, Dict]]:
    """Load persisted hypergraphs from disk. Returns None if cache is missing."""
    cache = Path(cache_dir)
    required = ["theme_hypergraph.json", "entity_hypergraph.json", "chunk_registry.pkl"]
    if not all((cache / f).exists() for f in required):
        return None

    def dict_to_hg(data: Dict) -> HyperGraph:
        hg = HyperGraph(
            entity_to_hedges=data["entity_to_hedges"],
            chunk_to_hedges=data["chunk_to_hedges"],
            hedge_id_list=data["hedge_id_list"],
        )
        for hid, hdata in data["hedges"].items():
            hg.hedges[hid] = HyperEdge(
                hedge_id    = hdata["hedge_id"],
                entities    = hdata["entities"],
                description = hdata["description"],
                chunk_ids   = hdata["chunk_ids"],
                hedge_type  = hdata["hedge_type"],
                embedding   = hdata.get("embedding"),
            )
        hg.build_embedding_matrix()
        return hg

    try:
        with open(cache / "theme_hypergraph.json") as f:
            theme_data = json.load(f)
        with open(cache / "entity_hypergraph.json") as f:
            entity_data = json.load(f)
        with open(cache / "chunk_registry.pkl", "rb") as f:
            chunk_registry = pickle.load(f)

        logger.info("Loaded hypergraphs from cache: %s", cache_dir)
        return dict_to_hg(theme_data), dict_to_hg(entity_data), chunk_registry
    except Exception as exc:
        logger.warning("Hypergraph cache load failed: %s", exc)
        return None



In [28]:


# ===========================================================================
# SECTION 5 -- RETRIEVAL UTILITIES
# ===========================================================================

def cosine_sim_batch(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarities between a query vector and a matrix of vectors.
    Both query_vec and matrix rows are assumed to be L2-normalized (BGE-large
    uses normalize_embeddings=True, so inner product == cosine similarity).

    Args:
        query_vec : shape (D,) -- the query embedding.
        matrix    : shape (N, D) -- the hyperedge embedding matrix.

    Returns:
        np.array of shape (N,) -- cosine similarities.
    """
    return (matrix @ query_vec).astype(np.float32)


def retrieve_theme_hyperedges(
    query: str,
    theme_keywords: List[str],
    theme_hg: HyperGraph,
    embeddings: HuggingFaceEmbeddings,
    config: CogRAGConfig,
) -> Tuple[List[str], Set[str]]:
    """
    Stage 1 retrieval: activate relevant theme hyperedges + 1-hop diffusion.

    Two-step process:
    1. KEYWORD MATCHING: For each theme keyword in theme_keywords, find theme
       hyperedges whose entities or description contain the keyword (case-insensitive).
       If keyword matches are found, score by number of keyword matches.

    2. SEMANTIC SIMILARITY FALLBACK: For any remaining spots in THEME_TOP_K,
       fill with highest-cosine-sim hyperedges from the embedding matrix.
       This handles queries where the exact keywords don't match theme labels
       but the semantic content is relevant (paraphrase, domain shift).

    3. DIFFUSION: From the retrieved set, expand to 1-hop neighbors sharing
       any entity vertex. This models Cog-RAG's "contextual activation spread."

    Args:
        query           : User query string.
        theme_keywords  : Keywords extracted from query for theme matching.
        theme_hg        : The built theme hypergraph.
        embeddings      : HuggingFaceEmbeddings for semantic search.
        config          : CogRAGConfig.

    Returns:
        Tuple of (top-k theme hedge_ids, diffused neighbor hedge_ids).
    """
    if theme_hg.n_hedges() == 0:
        return [], set()

    # --- Step 1: Keyword matching ----------------------------------------
    keyword_scores: Dict[str, int] = {}
    for kw in theme_keywords:
        kw_lower = kw.lower()
        # Match against entity names
        for ent, hids in theme_hg.entity_to_hedges.items():
            if kw_lower in ent:
                for hid in hids:
                    keyword_scores[hid] = keyword_scores.get(hid, 0) + 1
        # Match against descriptions
        for hid, hedge in theme_hg.hedges.items():
            if kw_lower in hedge.description.lower():
                keyword_scores[hid] = keyword_scores.get(hid, 0) + 1

    kw_ranked = sorted(keyword_scores, key=lambda k: keyword_scores[k], reverse=True)
    selected  = kw_ranked[: config.THEME_TOP_K]

    # --- Step 2: Semantic fallback to fill remaining slots ----------------
    n_remaining = config.THEME_TOP_K - len(selected)
    if n_remaining > 0 and theme_hg.hedge_embeddings is not None:
        query_vec = np.array(
            embeddings.embed_query(query), dtype=np.float32
        )
        sims  = cosine_sim_batch(query_vec, theme_hg.hedge_embeddings)
        order = np.argsort(sims)[::-1]
        for idx in order:
            hid = theme_hg.hedge_id_list[idx]
            if hid not in selected:
                selected.append(hid)
                n_remaining -= 1
                if n_remaining == 0:
                    break

    # --- Step 3: 1-hop diffusion ----------------------------------------
    diffused = set()
    for _ in range(config.DIFFUSION_HOPS):
        diffused |= theme_hg.neighbor_hedges(selected)

    return selected[: config.THEME_TOP_K], diffused


In [29]:

def retrieve_entity_hyperedges(
    query: str,
    entity_keywords: List[str],
    theme_answer: str,
    entity_hg: HyperGraph,
    embeddings: HuggingFaceEmbeddings,
    config: CogRAGConfig,
) -> List[str]:
    """
    Stage 2 retrieval: fine-grained entity hyperedge recall guided by Stage 1.

    Scoring formula per entity hyperedge e:
        coverage = |{ek in entity_keywords : ek in e.entities_lower}| / len(entity_keywords)
        sem_sim  = cosine(embed(query + " " + theme_answer[:100]), embed(e.description))
        score(e) = ENTITY_COVERAGE_WEIGHT * coverage + SEMANTIC_WEIGHT * sem_sim

    The theme_answer from Stage 1 is concatenated to the query for the semantic
    similarity computation. This is the ALIGNMENT mechanism of Cog-RAG: Stage 2
    retrieval is GUIDED by Stage 1's thematic understanding.

    Without theme guidance (entity-only retrieval): sem_sim uses raw query embedding.
    With theme guidance (full Cog-RAG Config 5): sem_sim uses (query + theme_answer)
    embedding, which encodes BOTH the query intent AND the thematic context from Stage 1.
    This alignment ensures Stage 2 retrieves fine-grained facts that are COHERENT
    with the macro-thematic context, not just individually relevant to raw query keywords.

    Args:
        query           : User query string.
        entity_keywords : Entity keywords extracted from query.
        theme_answer    : Preliminary thematic answer from Stage 1 (empty if no Stage 1).
        entity_hg       : The built entity hypergraph.
        embeddings      : HuggingFaceEmbeddings.
        config          : CogRAGConfig.

    Returns:
        List of top-ENTITY_TOP_K entity hedge_ids.
    """
    if entity_hg.n_hedges() == 0:
        return []

    # Build guided search query
    guided_query = query
    if theme_answer:
        guided_query = query + " " + theme_answer[:150]

    query_vec = np.array(embeddings.embed_query(guided_query), dtype=np.float32)

    # Compute semantic similarities for all entity hyperedges
    sims: Dict[str, float] = {}
    if entity_hg.hedge_embeddings is not None:
        sim_array = cosine_sim_batch(query_vec, entity_hg.hedge_embeddings)
        for i, hid in enumerate(entity_hg.hedge_id_list):
            sims[hid] = float(sim_array[i])

    # Compute entity keyword coverage for all hyperedges
    ek_lower = [ek.lower().strip() for ek in entity_keywords]
    scores: Dict[str, float] = {}
    for hid, hedge in entity_hg.hedges.items():
        ents_lower = [e.lower() for e in hedge.entities]
        if ek_lower:
            coverage = sum(1 for ek in ek_lower
                           if any(ek in e for e in ents_lower)) / len(ek_lower)
        else:
            coverage = 0.0
        sem_sim = sims.get(hid, 0.0)
        scores[hid] = (config.ENTITY_COVERAGE_WEIGHT * coverage
                       + config.SEMANTIC_WEIGHT * sem_sim)

    ranked = sorted(scores, key=lambda k: scores[k], reverse=True)
    return ranked[: config.ENTITY_TOP_K]


In [30]:


def assemble_context_from_hedges(
    hedge_ids:      List[str],
    hg:             HyperGraph,
    chunk_registry: Dict[str, Document],
    max_chars:      int,
    include_hedge_text: bool = True,
) -> str:
    """
    Assemble a context string from hyperedge IDs.

    For each retrieved hedge:
        - Include the hyperedge description (the extracted relational fact / theme label).
        - Include the source chunk text (the original passage from the PDF).
    Deduplicate chunks: if two hyperedges point to the same chunk, include the chunk once.
    Truncate total context to max_chars.

    Args:
        hedge_ids           : Ordered list of hedge IDs to include.
        hg                  : The hypergraph (theme or entity).
        chunk_registry      : Dict[chunk_id -> Document].
        max_chars           : Maximum total characters for the context.
        include_hedge_text  : Whether to include the hyperedge description.

    Returns:
        Assembled context string.
    """
    seen_chunks:  Set[str] = set()
    context_parts: List[str] = []
    total_chars = 0

    for hid in hedge_ids:
        hedge = hg.hedges.get(hid)
        if not hedge:
            continue

        part_lines = []
        if include_hedge_text:
            entities_str = ", ".join(hedge.entities[:4])
            part_lines.append(f"[{hedge.hedge_type.upper()}] {hedge.description}"
                               f" | entities: {entities_str}")

        for cid in hedge.chunk_ids:
            if cid in seen_chunks:
                continue
            seen_chunks.add(cid)
            doc = chunk_registry.get(cid)
            if doc:
                paper = doc.metadata.get("paper_name", "?")[:25]
                page  = doc.metadata.get("page", "?")
                part_lines.append(f"[{paper} p{page}] {doc.page_content.strip()}")

        part = "\n".join(part_lines)
        if total_chars + len(part) > max_chars:
            remaining = max_chars - total_chars
            if remaining > 100:
                context_parts.append(part[:remaining])
            break
        context_parts.append(part)
        total_chars += len(part)

    return "\n\n---\n\n".join(context_parts)



In [31]:

# ===========================================================================
# SECTION 6 -- FIVE COG-RAG CONFIGURATIONS
# ===========================================================================

# ---------------------------------------------------------------------------
# CONFIG 1: Baseline -- flat FAISS retrieval
# ---------------------------------------------------------------------------

def run_config1_baseline(
    question: str, flat_vs: FAISS,
    llm: ChatOllama, config: CogRAGConfig,
) -> CogRAGTrace:
    """
    Configuration 1: Standard flat FAISS retrieval (no hypergraph).
    Top-FLAT_TOP_K by cosine similarity. Control condition.
    """
    trace = CogRAGTrace(question=question, strategy="Config1_Baseline_FlatRAG")
    t0 = time.perf_counter()

    t_ret = time.perf_counter()
    docs  = flat_vs.similarity_search(question, k=config.FLAT_TOP_K)
    trace.retrieval_ms = (time.perf_counter() - t_ret) * 1000

    context = "\n\n---\n\n".join([
        f"[{d.metadata.get('paper_name','?')[:25]} p{d.metadata.get('page','?')}]\n"
        f"{d.page_content.strip()}"
        for d in docs
    ])
    prompt = ChatPromptTemplate.from_template(config.P_BASELINE_ANSWER)
    t_gen  = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.final_answer  = answer
    trace.llm_calls     = 1
    trace.total_ms      = (time.perf_counter() - t0) * 1000
    return trace


In [32]:

# ---------------------------------------------------------------------------
# CONFIG 2: Entity Hypergraph Only
# ---------------------------------------------------------------------------

def run_config2_entity_hypergraph(
    question: str, entity_hg: HyperGraph, chunk_registry: Dict[str, Document],
    embeddings: HuggingFaceEmbeddings, llm: ChatOllama, config: CogRAGConfig,
) -> CogRAGTrace:
    """
    Configuration 2: Entity Hypergraph-only retrieval (no theme layer).

    Retrieves entity hyperedges directly from query keywords + semantic similarity.
    No Stage 1 theme activation. No theme guidance for Stage 2.
    Demonstrates the value of high-order entity modeling without the cognitive ordering.
    Ablation counterpart: compares to Config 4/5 to show the value of the theme layer.
    """
    trace = CogRAGTrace(question=question, strategy="Config2_EntityHypergraph_Only")
    t0 = time.perf_counter()

    # Extract entity keywords directly from query (no LLM call: simple token extraction)
    entity_keywords = [w.strip() for w in question.replace("?", "").split()
                       if len(w) > 3 and w[0].isupper()][:5]
    if not entity_keywords:
        entity_keywords = question.split()[:5]
    trace.entity_keywords = entity_keywords

    t_ret = time.perf_counter()
    top_entity_ids = retrieve_entity_hyperedges(
        query=question, entity_keywords=entity_keywords, theme_answer="",
        entity_hg=entity_hg, embeddings=embeddings, config=config,
    )
    trace.retrieval_ms         = (time.perf_counter() - t_ret) * 1000
    trace.stage2_entity_hedges = top_entity_ids

    entity_context = assemble_context_from_hedges(
        top_entity_ids, entity_hg, chunk_registry, config.MAX_CONTEXT_CHARS
    )
    if not entity_context:
        entity_context = "No entity hyperedges retrieved."

    prompt = ChatPromptTemplate.from_template(config.P_BASELINE_ANSWER)
    t_gen  = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": entity_context, "question": question}
    )
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.final_answer  = answer
    trace.llm_calls     = 1
    trace.total_ms      = (time.perf_counter() - t0) * 1000
    return trace



In [33]:

# ---------------------------------------------------------------------------
# CONFIG 3: Theme Hypergraph Only
# ---------------------------------------------------------------------------

def run_config3_theme_hypergraph(
    question: str, theme_hg: HyperGraph, chunk_registry: Dict[str, Document],
    embeddings: HuggingFaceEmbeddings, llm: ChatOllama, config: CogRAGConfig,
) -> CogRAGTrace:
    """
    Configuration 3: Theme Hypergraph-only retrieval (no entity layer).

    Activates theme hyperedges + 1-hop diffusion from theme keywords.
    No entity-level fine-grained facts. No two-stage cognitive ordering.
    Demonstrates the value of global thematic organization without micro-detail.
    Ablation counterpart: compares to Config 4/5 to show the value of the entity layer.
    Per Cog-RAG ablation study (Hu et al., 2025): removing the entity hypergraph
    causes the largest performance drop of all ablated components.
    """
    trace = CogRAGTrace(question=question, strategy="Config3_ThemeHypergraph_Only")
    t0 = time.perf_counter()

    # Simple keyword extraction from query (heuristic, no LLM for keyword-only config)
    theme_keywords = [w.strip() for w in question.replace("?", "").split()
                      if len(w) > 4][:4]
    trace.theme_keywords = theme_keywords

    t_ret = time.perf_counter()
    top_theme_ids, diffused = retrieve_theme_hyperedges(
        query=question, theme_keywords=theme_keywords,
        theme_hg=theme_hg, embeddings=embeddings, config=config,
    )
    trace.retrieval_ms       = (time.perf_counter() - t_ret) * 1000
    trace.stage1_theme_hedges = top_theme_ids
    trace.stage1_diffused    = diffused

    all_theme_ids = top_theme_ids + list(diffused)[:3]
    theme_context = assemble_context_from_hedges(
        all_theme_ids, theme_hg, chunk_registry, config.MAX_CONTEXT_CHARS
    )
    if not theme_context:
        theme_context = "No theme hyperedges retrieved."

    prompt = ChatPromptTemplate.from_template(config.P_BASELINE_ANSWER)
    t_gen  = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": theme_context, "question": question}
    )
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.final_answer  = answer
    trace.llm_calls     = 1
    trace.total_ms      = (time.perf_counter() - t0) * 1000
    return trace



In [34]:

# ---------------------------------------------------------------------------
# CONFIG 4: Full Dual-Hypergraph (flat fusion, no cognitive ordering)
# ---------------------------------------------------------------------------

def run_config4_dual_hypergraph_flat(
    question:       str,
    theme_hg:       HyperGraph,
    entity_hg:      HyperGraph,
    chunk_registry: Dict[str, Document],
    embeddings:     HuggingFaceEmbeddings,
    llm:            ChatOllama,
    config:         CogRAGConfig,
) -> CogRAGTrace:
    """
    Configuration 4: Full Dual-Hypergraph with FLAT score fusion (no two-stage ordering).

    Both theme and entity hyperedges are retrieved simultaneously using the raw query
    (no theme-aware guidance for entity retrieval). Contexts are concatenated and
    fed to the LLM in a single generation call.

    Ablation of the COGNITIVE ORDERING component:
        Config 4 uses both hypergraphs (same structural richness as Config 5) but
        REMOVES the two-stage sequential reasoning structure where Stage 1's thematic
        answer guides Stage 2's entity retrieval. This isolates the contribution of
        the cognitive ordering vs. the dual-hypergraph structure.

    Per Cog-RAG ablation results (Hu et al., 2025):
        Removing the two-stage retrieval strategy (this config) reduces performance
        by a measurable margin relative to Config 5 on complex queries, confirming
        that the cognitive ordering contributes independent value beyond the
        dual-hypergraph structure alone. The gap is largest on intra-domain dense
        corpora (e.g., medical) where thematic alignment is most critical.

    LLM calls: 2 (1 for keyword extraction + 1 for answer generation).
    """
    trace = CogRAGTrace(question=question,
                        strategy="Config4_DualHypergraph_FlatFusion")
    t0 = time.perf_counter()

    # --- LLM-based keyword extraction (unified for both graphs) -----------
    kw_prompt = config.P_KEYWORD_EXTRACT.format(query=question)
    t_kw = time.perf_counter()
    raw_kw = _llm_call(kw_prompt, llm)
    kw_ms  = (time.perf_counter() - t_kw) * 1000
    parsed_kw = _safe_json_parse(raw_kw, {"theme_keywords": [], "entity_keywords": []})
    theme_kws  = parsed_kw.get("theme_keywords", [])[:3]
    entity_kws = parsed_kw.get("entity_keywords", [])[:5]
    trace.theme_keywords  = theme_kws
    trace.entity_keywords = entity_kws

    # --- Retrieve from BOTH hypergraphs in parallel (flat, no ordering) ---
    t_ret = time.perf_counter()
    top_theme_ids, diffused = retrieve_theme_hyperedges(
        question, theme_kws, theme_hg, embeddings, config
    )
    top_entity_ids = retrieve_entity_hyperedges(
        question, entity_kws, theme_answer="",   # NO theme guidance
        entity_hg=entity_hg, embeddings=embeddings, config=config,
    )
    trace.retrieval_ms         = (time.perf_counter() - t_ret) * 1000
    trace.stage1_theme_hedges  = top_theme_ids
    trace.stage1_diffused      = diffused
    trace.stage2_entity_hedges = top_entity_ids

    # Assemble flat combined context
    half_chars = config.MAX_CONTEXT_CHARS // 2
    theme_ctx  = assemble_context_from_hedges(
        top_theme_ids + list(diffused)[:2], theme_hg, chunk_registry, half_chars
    )
    entity_ctx = assemble_context_from_hedges(
        top_entity_ids, entity_hg, chunk_registry, half_chars
    )
    combined_ctx = (theme_ctx + "\n\n--- Entity Facts ---\n\n" + entity_ctx
                    ) if entity_ctx else theme_ctx

    prompt = ChatPromptTemplate.from_template(config.P_BASELINE_ANSWER)
    t_gen  = time.perf_counter()
    answer = (prompt | llm | StrOutputParser()).invoke(
        {"context": combined_ctx, "question": question}
    )
    trace.generation_ms = (time.perf_counter() - t_gen) * 1000
    trace.final_answer  = answer
    trace.llm_calls     = 2
    trace.total_ms      = (time.perf_counter() - t0) * 1000
    return trace



In [35]:

# ---------------------------------------------------------------------------
# CONFIG 5: Full Cog-RAG Two-Stage Cognitive Retrieval [BEST]
# ---------------------------------------------------------------------------

def run_config5_full_cograg(
    question:       str,
    theme_hg:       HyperGraph,
    entity_hg:      HyperGraph,
    chunk_registry: Dict[str, Document],
    embeddings:     HuggingFaceEmbeddings,
    llm:            ChatOllama,
    config:         CogRAGConfig,
) -> CogRAGTrace:
    """
    Configuration 5: Full Cog-RAG Two-Stage Cognitive Retrieval [BEST].

    Implements the complete Cog-RAG algorithm from Hu et al. (2025) arXiv 2511.13201:

    STAGE 1 -- Theme Hypergraph Activation (top-down macro reasoning):
        1a. Extract theme_keywords and entity_keywords from query via LLM.
        1b. Retrieve top-THEME_TOP_K theme hyperedges (keyword match + semantic).
        1c. Perform 1-hop diffusion on theme hypergraph to activate neighbors.
        1d. Generate a PRELIMINARY THEMATIC ANSWER A_theme from theme context.
            A_theme: 3-5 sentences capturing the macro-thematic understanding.

    STAGE 2 -- Entity Hypergraph Recall (bottom-up micro reasoning guided by Stage 1):
        2a. Use A_theme as additional context for entity hyperedge retrieval.
            The semantic search query becomes: query + " " + A_theme[:150]
            This embeds BOTH the original query intent AND Stage 1's thematic guidance.
        2b. Score entity hyperedges: coverage * 0.6 + semantic_sim(guided) * 0.4
        2c. Retrieve top-ENTITY_TOP_K entity hyperedges aligned with Stage 1's themes.
        2d. Generate FINAL ANSWER from: theme context + entity context + A_theme refinement.

    The cognitive analogy:
        Stage 1 = "Top-down": You recognize the macro-topic before recalling details.
                  Like scanning a textbook chapter to understand its structure.
        Stage 2 = "Bottom-up guided": You recall specific facts within the activated theme.
                  Like finding a specific equation WITHIN the chapter you already recognized.

    Cog-RAG results (Hu et al., 2025):
        In intra-domain sparse datasets: outperforms HiRAG by 15.6% and 20.0%.
        In intra-domain dense medical corpora: improves over Hyper-RAG by 21.0% and 26.4%.
        Benefits from dual hypergraph + cognitive ordering: both are confirmed ablation.

    LLM calls: 3
        1. Keyword extraction: P_KEYWORD_EXTRACT
        2. Stage 1 preliminary answer: P_THEME_ANSWER
        3. Stage 2 final answer: P_FINAL_ANSWER

    Args:
        question       : User query.
        theme_hg       : Built and embedded theme hypergraph.
        entity_hg      : Built and embedded entity hypergraph.
        chunk_registry : Dict[chunk_id -> Document].
        embeddings     : HuggingFaceEmbeddings.
        llm            : ChatOllama.
        config         : CogRAGConfig.

    Returns:
        CogRAGTrace with full two-stage audit, keywords, and preliminary answer.
    """
    trace = CogRAGTrace(question=question,
                        strategy="Config5_Full_CogRAG_TwoStage [BEST]")
    t0 = time.perf_counter()

    # --- LLM Call 1: Keyword Extraction -----------------------------------
    kw_prompt = config.P_KEYWORD_EXTRACT.format(query=question)
    raw_kw    = _llm_call(kw_prompt, llm)
    parsed_kw = _safe_json_parse(raw_kw, {"theme_keywords": [], "entity_keywords": []})
    theme_kws  = parsed_kw.get("theme_keywords",  [])[:3]
    entity_kws = parsed_kw.get("entity_keywords", [])[:5]
    trace.theme_keywords  = theme_kws
    trace.entity_keywords = entity_kws

    logger.info(
        "Config5 CogRAG: theme_kws=%s  entity_kws=%s",
        theme_kws, entity_kws
    )

    # --- STAGE 1: Theme Hypergraph Activation ----------------------------
    t_stage1 = time.perf_counter()
    top_theme_ids, diffused = retrieve_theme_hyperedges(
        query=question, theme_keywords=theme_kws,
        theme_hg=theme_hg, embeddings=embeddings, config=config,
    )
    trace.stage1_theme_hedges = top_theme_ids
    trace.stage1_diffused     = diffused

    # Build Stage 1 context (theme hyperedges + diffused neighbors)
    all_theme_ids = top_theme_ids + list(diffused)[:3]
    theme_ctx = assemble_context_from_hedges(
        all_theme_ids, theme_hg, chunk_registry,
        max_chars=config.MAX_CONTEXT_CHARS // 2,
    )
    stage1_ms = (time.perf_counter() - t_stage1) * 1000

    # --- LLM Call 2: Preliminary Thematic Answer -------------------------
    theme_prompt = config.P_THEME_ANSWER.format(
        theme_context=theme_ctx or "No theme context available.",
        question=question,
    )
    t_theme_gen   = time.perf_counter()
    theme_answer  = _llm_call(theme_prompt, llm)
    theme_gen_ms  = (time.perf_counter() - t_theme_gen) * 1000
    trace.theme_answer = theme_answer

    logger.info(
        "Config5 Stage1: %d theme hedges + %d diffused  "
        "(retrieval=%.0fms, preliminary_gen=%.0fms)",
        len(top_theme_ids), len(diffused), stage1_ms, theme_gen_ms,
    )

    # --- STAGE 2: Entity Hypergraph Guided by Stage 1 -------------------
    t_stage2 = time.perf_counter()
    top_entity_ids = retrieve_entity_hyperedges(
        query=question, entity_keywords=entity_kws,
        theme_answer=theme_answer,   # THEME-GUIDED SEARCH
        entity_hg=entity_hg, embeddings=embeddings, config=config,
    )
    entity_ctx = assemble_context_from_hedges(
        top_entity_ids, entity_hg, chunk_registry,
        max_chars=config.MAX_CONTEXT_CHARS // 2,
    )
    stage2_ms = (time.perf_counter() - t_stage2) * 1000
    trace.stage2_entity_hedges = top_entity_ids
    trace.retrieval_ms = stage1_ms + stage2_ms

    logger.info(
        "Config5 Stage2: %d entity hedges (retrieval=%.0fms)",
        len(top_entity_ids), stage2_ms,
    )

    # --- LLM Call 3: Final Answer fusing theme + entity + preliminary ----
    final_prompt = config.P_FINAL_ANSWER.format(
        theme_context  = theme_ctx or "No theme context available.",
        entity_context = entity_ctx or "No entity context available.",
        theme_answer   = theme_answer[:400],
        question       = question,
    )
    t_final_gen = time.perf_counter()
    final_answer = _llm_call(final_prompt, llm)
    trace.generation_ms = (time.perf_counter() - t_final_gen) * 1000 + theme_gen_ms
    trace.final_answer  = final_answer
    trace.llm_calls     = 3
    trace.total_ms      = (time.perf_counter() - t0) * 1000

    logger.info(
        "Config5 Full CogRAG complete: total=%.0fms, llm_calls=3",
        trace.total_ms,
    )
    return trace


In [36]:

# ===========================================================================
# SECTION 7 -- COMPARATIVE RUNNER
# ===========================================================================

def run_all_configs(
    question:       str,
    flat_vs:        FAISS,
    theme_hg:       HyperGraph,
    entity_hg:      HyperGraph,
    chunk_registry: Dict[str, Document],
    embeddings:     HuggingFaceEmbeddings,
    llm:            ChatOllama,
    config:         CogRAGConfig,
) -> Dict[str, CogRAGTrace]:
    print("\n" + "#" * 78)
    print(f"QUERY: {question}")
    print("#" * 78)

    runners = {
        "Config1_Baseline_FlatRAG":              lambda q: run_config1_baseline(
            q, flat_vs, llm, config),
        "Config2_EntityHypergraph_Only":         lambda q: run_config2_entity_hypergraph(
            q, entity_hg, chunk_registry, embeddings, llm, config),
        "Config3_ThemeHypergraph_Only":          lambda q: run_config3_theme_hypergraph(
            q, theme_hg, chunk_registry, embeddings, llm, config),
        "Config4_DualHypergraph_FlatFusion":     lambda q: run_config4_dual_hypergraph_flat(
            q, theme_hg, entity_hg, chunk_registry, embeddings, llm, config),
        "Config5_Full_CogRAG_TwoStage [BEST]":  lambda q: run_config5_full_cograg(
            q, theme_hg, entity_hg, chunk_registry, embeddings, llm, config),
    }

    traces: Dict[str, CogRAGTrace] = {}
    for label, fn in runners.items():
        print(f"\n{'='*62}\nRUNNING: {label}\n{'='*62}")
        try:
            trace = fn(question)
            trace.print_trace()
            traces[label] = trace
        except Exception as exc:
            logger.error("Config %s failed: %s", label, exc, exc_info=True)

    print("\n" + "=" * 78)
    print("COG-RAG COMPARISON SUMMARY")
    print(f"{'Config':<48} {'LLMCalls':>9} {'Retr_ms':>8} {'Gen_ms':>8} {'Total_ms':>9}")
    print("-" * 78)
    for lbl, tr in traces.items():
        print(
            f"{lbl:<48} {tr.llm_calls:>9} "
            f"{tr.retrieval_ms:>8.0f} {tr.generation_ms:>8.0f} {tr.total_ms:>9.0f}"
        )
    print("=" * 78 + "\n")
    return traces



In [37]:
"""
    End-to-end Cog-RAG pipeline orchestrator.

    INDEXING PHASE (run once, then cached to INDEX_CACHE_DIR):
        - Load + chunk 3 arXiv PDFs
        - For each chunk: LLM theme extraction + entity extraction (~2 calls/chunk)
        - Embed all hyperedge descriptions with BGE-large
        - Persist to disk as JSON + pkl

    QUERY PHASE (each query):
        Config 1: 1 LLM call
        Config 2: 1 LLM call
        Config 3: 1 LLM call
        Config 4: 2 LLM calls (keyword extraction + answer)
        Config 5: 3 LLM calls (keyword + theme prelim + final answer)

    PRODUCTION NOTE:
        The indexing phase requires ~2,400 LLM calls for 1,200 chunks.
        To reduce indexing cost: use gpt-4o-mini for extraction (theme + entity)
        and gpt-4o for answer generation only (Config 5 calls 2 and 3).
        Override config.OLLAMA_MODEL to "gpt-4o-mini" for indexing,
        then switch to "gpt-4o" for query-time generation.
    """

'\n    End-to-end Cog-RAG pipeline orchestrator.\n\n    INDEXING PHASE (run once, then cached to INDEX_CACHE_DIR):\n        - Load + chunk 3 arXiv PDFs\n        - For each chunk: LLM theme extraction + entity extraction (~2 calls/chunk)\n        - Embed all hyperedge descriptions with BGE-large\n        - Persist to disk as JSON + pkl\n\n    QUERY PHASE (each query):\n        Config 1: 1 LLM call\n        Config 2: 1 LLM call\n        Config 3: 1 LLM call\n        Config 4: 2 LLM calls (keyword extraction + answer)\n        Config 5: 3 LLM calls (keyword + theme prelim + final answer)\n\n    PRODUCTION NOTE:\n        The indexing phase requires ~2,400 LLM calls for 1,200 chunks.\n        To reduce indexing cost: use gpt-4o-mini for extraction (theme + entity)\n        and gpt-4o for answer generation only (Config 5 calls 2 and 3).\n        Override config.OLLAMA_MODEL to "gpt-4o-mini" for indexing,\n        then switch to "gpt-4o" for query-time generation.\n    '

In [38]:
config = CogRAGConfig()
logger.info("=== Cog-RAG Dual-Hypergraph Pipeline Starting ===")


2026-05-25 21:53:06 | INFO     | cograg | === Cog-RAG Dual-Hypergraph Pipeline Starting ===


In [39]:
pdf_paths  = download_pdfs(config)


2026-05-25 21:53:06 | INFO     | cograg | Cached: attention_is_all_you_need.pdf
2026-05-25 21:53:06 | INFO     | cograg | Cached: bert_pretraining_transformers.pdf
2026-05-25 21:53:06 | INFO     | cograg | Cached: rag_knowledge_intensive_nlp.pdf


In [40]:
chunks     = load_and_chunk_documents(pdf_paths, config)


2026-05-25 21:53:08 | INFO     | cograg |   attention_is_all_you_need.pdf -> 104 chunks
2026-05-25 21:53:09 | INFO     | cograg |   bert_pretraining_transformers.pdf -> 173 chunks
2026-05-25 21:53:09 | INFO     | cograg |   rag_knowledge_intensive_nlp.pdf -> 181 chunks
2026-05-25 21:53:09 | INFO     | cograg | Total chunks: 458


In [41]:
embeddings = build_bge_embeddings(config)


2026-05-25 21:53:09 | INFO     | cograg | Loading BGE: BAAI/bge-large-en-v1.5
2026-05-25 21:53:09 | INFO     | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5
2026-05-25 21:53:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 21:53:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-05-25 21:53:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 21:53:10 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3966.23it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-25 21:53:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 21:53:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config.json "HTTP/1.1 200 OK"
2026-05-25 21:53:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-25 21:53:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-25 21:53:13 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-05-25 21:53:13 | INFO     | httpx |

In [42]:
flat_vs    = build_flat_faiss(chunks, embeddings, config)


2026-05-25 21:53:14 | INFO     | cograg | Flat FAISS loaded. Vectors: 458


In [43]:
llm        = build_ollama_llm(config)


In [ ]:
# --- Try loading cached hypergraphs; build if missing -----------------
cached = load_hypergraphs(config.INDEX_CACHE_DIR)
if cached is not None:
    theme_hg, entity_hg, chunk_registry = cached
    logger.info(
        "Loaded from cache: theme=%d hedges, entity=%d hedges",
        theme_hg.n_hedges(), entity_hg.n_hedges(),
    )
else:
    logger.info("No cache found -- building dual hypergraph (this will take a while) ...")
    t_idx = time.perf_counter()
    theme_hg, entity_hg, chunk_registry = build_dual_hypergraph(
        chunks, embeddings, llm, config
    )
    idx_ms = (time.perf_counter() - t_idx) * 1000
    logger.info("Indexing complete in %.0f ms. Saving to cache ...", idx_ms)
    save_hypergraphs(theme_hg, entity_hg, chunk_registry, config.INDEX_CACHE_DIR)


2026-05-25 22:13:30 | INFO     | cograg | No cache found -- building dual hypergraph (this will take a while) ...
2026-05-25 22:13:30 | INFO     | cograg | Building dual hypergraph from 458 chunks ...
2026-05-25 22:13:30 | INFO     | cograg | WARNING: This requires ~916 LLM calls. Run once and cache.
2026-05-25 22:13:30 | INFO     | cograg |   Processing chunk 0 / 458 ...
2026-05-25 22:13:33 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:13:40 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:14:22 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:14:29 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:15:02 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-25 22:15:10 | INFO     | httpx | HTTP Request: POST http://localhos

In [ ]:

demo_questions = [
    # Multi-entity: needs high-order entity hyperedge (n=4+)
    "What are the specific parameter values for the Transformer base model including d_model, h, d_k, d_v, and d_ff?",

    # Thematic: requires global theme activation (spans multiple chunks)
    "What is the overall architecture of the Transformer and how do the encoder and decoder layers differ?",

    # Cross-paper: thematic organization helps connect across documents
    "How does BERT differ from the original Transformer in its use of pre-training and the encoder structure?",

    # Complex relational: pure entity hyperedge query
    "What are the relationships between attention heads, key dimension, value dimension, and model dimension in multi-head attention?",

    # Cognitive ordering test: theme answer should guide entity retrieval
    "How does the Transformer handle variable-length sequences without recurrence, and what mechanisms replace positional dependencies?",
]



In [ ]:
for question in demo_questions:
    run_all_configs(
        question, flat_vs, theme_hg, entity_hg,
        chunk_registry, embeddings, llm, config,
    )

logger.info("=== Cog-RAG Pipeline Demo Complete ===")


In [ ]:
test_question = "What are the specific parameter values for the Transformer base model including d_model, h, d_k, d_v, and d_ff?"

baseline_trace = run_config1_baseline(
    test_question,
    flat_vs,
    llm,
    config,
)

print("TEST QUERY:", test_question)
print("\nBASELINE OUTPUT:\n", baseline_trace.final_answer)
print("\nMETRICS:", {
    "llm_calls": baseline_trace.llm_calls,
    "retrieval_ms": round(baseline_trace.retrieval_ms, 2),
    "generation_ms": round(baseline_trace.generation_ms, 2),
    "total_ms": round(baseline_trace.total_ms, 2),
})

2026-05-25 21:40:35 | INFO     | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
TEST QUERY: What are the specific parameter values for the Transformer base model including d_model, h, d_k, d_v, and d_ff?

BASELINE OUTPUT:
 The provided documents do not contain sufficient information to answer this question.

METRICS: {'llm_calls': 1, 'retrieval_ms': 384.09, 'generation_ms': 15093.35, 'total_ms': 15483.36}
